### Retrieval Stage

In [1]:
!pip install -qq cornac==2.3.5

import ast
from tqdm import tqdm
import numpy as np
import pandas as pd

from cornac.models import GRU4Rec
from cornac.data.dataset import SequentialDataset
from cornac.models.gru4rec.gru4rec import GRU4RecModel


root = "/kaggle/input/rental-product-recommendation-system"


_original_init = GRU4RecModel._init_numpy_weights

def _patched_init_numpy_weights(self, shape):
    weights = _original_init(self, shape)
    return weights.astype(np.float32)

GRU4RecModel._init_numpy_weights = _patched_init_numpy_weights

def get_old_to_new_prod_ids():
    old_to_new_prod_id = pd.read_csv(f"{root}/old_site_new_site_products.csv", dtype=str)
    old_to_new_prod_id = old_to_new_prod_id.set_index("old_site_id")["new_site_id"].to_dict()
    return old_to_new_prod_id


def get_slug_to_ids():
    old_to_new_prod_id = get_old_to_new_prod_ids()

    df_prod_old = pd.read_csv(
        f"{root}/old_site_products.csv",
        usecols=["id", "slug"],
        dtype=str,
    )
    df_prod_new = pd.read_csv(
        f"{root}/new_site_products.csv",
        usecols=["id", "slug"],
        dtype=str,
    )

    df_prod_old["id"] = df_prod_old["id"].map(old_to_new_prod_id)
    df_prod_old = df_prod_old.dropna(subset=["id"])

    df_prod = pd.concat([df_prod_new, df_prod_old], axis=0)
    df_prod = df_prod.drop_duplicates(subset=["id", "slug"])

    slug_to_id = df_prod.set_index("slug")["id"].to_dict()

    return slug_to_id


def predict_next(model, user_ids, historical_item_list, k, allowed_product_ids=None):
    rev_iid_map = {v: k for k, v in model.iid_map.items()}
    num_items = len(rev_iid_map)
    global_mask = None
    if allowed_product_ids is not None:
        allowed_internal_ids = [
            model.iid_map[iid]
            for iid in allowed_product_ids
            if iid in model.iid_map
        ]

        global_mask = np.ones(num_items, dtype=bool)
        global_mask[allowed_internal_ids] = False

    recom_items, recom_scores = [], []
    for user_id, history_items in tqdm(zip(user_ids, historical_item_list), total=len(user_ids)):
        history_internal_idxs = [model.iid_map[x] for x in history_items if x in model.iid_map]

        scores = model.score(
            user_idx=model.uid_map[user_id],
            history_items=history_internal_idxs,
        )

        if global_mask is not None:
            scores[global_mask] = -np.inf

        scores[history_internal_idxs] = -np.inf

        if k < len(scores):
            unsorted_topk_idx = np.argpartition(scores, -k)[-k:]
            topk_idx_sorted = unsorted_topk_idx[np.argsort(scores[unsorted_topk_idx])][::-1]
        else:
            topk_idx_sorted = np.argsort(scores)[::-1]

        topk_vals = scores[topk_idx_sorted]
        topk_iid = [rev_iid_map[idx] for idx in topk_idx_sorted]

        recom_items.append(topk_iid)
        recom_scores.append(topk_vals.tolist())

    return recom_items, recom_scores


def get_hits_data():
    slug_to_ids = get_slug_to_ids()

    df_hits = pd.concat([
        pd.read_csv(
            f"{root}/metrika_hits.csv",
            usecols=['date_time', 'slug', 'page_type', "project_id", "is_page_view", "watch_id"],
            dtype=str
        ),
        pd.read_csv(
            f"{root}/metrika_hits_test.csv",
            usecols=['date_time', 'slug', 'page_type', "project_id", "is_page_view", "watch_id"],
            dtype=str
        ),
    ], ignore_index=True, axis=0)

    df_hits["date_time"] = pd.to_datetime(df_hits["date_time"], format="ISO8601")
    df_hits = df_hits[df_hits["is_page_view"].eq("1")]

    is_search = df_hits["page_type"].eq("SEARCH")
    is_cart = df_hits["page_type"].eq("CART")
    is_checkout = df_hits["page_type"].eq("CHECKOUT")
    is_order = df_hits["page_type"].eq("ORDER")
    is_unavailable = df_hits["page_type"].eq("UNAVAILABLE_PRODUCT")

    df_hits.loc[is_search, "slug"] = "search"
    df_hits.loc[is_cart, "slug"] = "cart"
    df_hits.loc[is_checkout, "slug"] = "checkout"
    df_hits.loc[is_order, "slug"] = "order"
    df_hits.loc[is_unavailable, "slug"] = "unavailable"

    df_hits = df_hits.dropna(subset=["slug"])
    df_hits["product_id"] = df_hits["slug"].map(slug_to_ids)
    missing_product_id = df_hits["product_id"].isnull()

    non_identified_slugs = df_hits.loc[
        missing_product_id, "slug"
    ].unique()

    new_mapper = {slug: str(500000000 + i) for i, slug in enumerate(non_identified_slugs)}
    slug_to_ids.update(new_mapper)

    df_hits["product_id"] = df_hits["slug"].map(slug_to_ids)
    return df_hits


def get_visits_data():
    df_visits = pd.concat([
        pd.read_csv(
            f"{root}/metrika_visits.csv",
            usecols=['client_id', 'visit_id', 'watch_ids'],
            dtype=str
        ),
        pd.read_csv(
            f"{root}/metrika_visits_test.csv",
            usecols=['client_id', 'visit_id', 'watch_ids'],
            dtype=str
        ),
    ], ignore_index=True, axis=0)

    df_visits["watch_ids"] = df_visits["watch_ids"].apply(lambda x: ast.literal_eval(x))
    df_visits = df_visits.explode("watch_ids")
    df_visits = df_visits.rename(columns={"watch_ids": "watch_id"})

    return df_visits


def create_recom_data(df_hits, df_visits):
    df_merged = pd.merge(df_hits, df_visits, on="watch_id", how="left")
    df_merged = pd.concat([
        df_merged[df_merged["page_type"].ne("PRODUCT")],
        df_merged[df_merged["page_type"].eq("PRODUCT")].drop_duplicates(["visit_id", "product_id"], keep="first")
    ])

    df_data = df_merged[["client_id", "visit_id", "product_id", "is_page_view", "page_type", "date_time", "slug", "project_id"]].dropna()
    df_data['date_time'] = df_data['date_time'].astype('int64') // 10 ** 9
    df_data = df_data.sort_values(['visit_id', 'date_time'])
    return df_data


def add_start_token(df):
    df_start = (
        df
        .groupby("visit_id").head(1)
        .assign(
            product_id=lambda d: np.where(d["project_id"] == "1", "000000000", "000000001"),
            page_type=lambda d: np.where(d["project_id"] == "1", "START_OLD", "START_NEW"),
            is_page_view="1",
        )
    )
    df_start["date_time"] = df_start["date_time"] - 1

    df = pd.concat([df, df_start], axis=0, ignore_index=True)
    df = df.sort_values(['visit_id', 'date_time'])
    return df


def get_data_splits(df_data, split_threshold):
    df_data = df_data.copy()
    df_data = df_data.sort_values(["visit_id", "date_time"])

    df_data['pos'] = df_data.groupby('visit_id', sort=False).cumcount() + 1
    df_data['total_items'] = df_data.groupby('visit_id')['product_id'].transform('count')
    df_data['is_train'] = df_data['pos'] <= (df_data['total_items'] * split_threshold).round()
    df_data.loc[df_data["project_id"].eq("1"), "is_train"] = True

    df_train = df_data[df_data['is_train']].copy()
    df_valid = df_data[~df_data['is_train']].copy()

    df_train = df_train.drop(
        columns=['pos', 'total_items', 'is_train']
    )
    df_valid = df_valid.drop(
        columns=['pos', 'total_items', 'is_train']
    )

    visit_ids = get_test_visit_ids()
    product_ids = get_allowed_product_ids()

    df_test = df_valid[
        df_valid["project_id"].eq("0") &
        df_valid["visit_id"].isin(visit_ids) &
        df_valid["product_id"].isin(product_ids)
    ]

    df_valid = df_valid[
        df_valid["project_id"].eq("0") &
        (~df_valid["visit_id"].isin(visit_ids)) &
        df_valid["product_id"].isin(product_ids)
    ]

    return df_train, df_valid, df_test


def get_test_visit_ids():
    return pd.read_csv(
        f"{root}/metrika_visits_test.csv",
        usecols=['visit_id'],
        dtype=str
    )["visit_id"].unique()


def get_allowed_product_ids():
    return pd.read_csv(
        f"{root}/new_site_products.csv",
        usecols=["id"],
        dtype=str,
    )["id"].unique()


def get_fitted_model(df_train, max_row_per_session):
    df_train = df_train.groupby('visit_id').head(max_row_per_session)

    train_data = SequentialDataset.build(
        list(df_train[[
            "client_id", 'visit_id', 'product_id', 'date_time'
        ]].itertuples(index=False, name=None)), fmt="USIT")

    model = GRU4Rec(
        layers=[100],
        loss="cross-entropy",
        n_sample=4096,
        dropout_p_embed=0.0,
        dropout_p_hidden=0.0,
        sample_alpha=0.0,
        batch_size=512,
        n_epochs=20,
        device="cuda",
        verbose=True,
        seed=123,
    )
    model.fit(train_data)
    return model


def create_submission(fitted_model, hist_data, visit_ids, k, allowed_product_ids):
    result = (
        hist_data[hist_data["visit_id"].isin(visit_ids)]
        .groupby("visit_id", sort=False)
        .agg(
            user_id=("client_id", "first"),
            historical_items=("product_id", lambda x: x.tolist())
        )
        .reset_index()
    )

    visit_ids = result["visit_id"].tolist()
    user_ids = result["user_id"].tolist()
    historical_item_list = result["historical_items"].tolist()

    recoms, scores = predict_next(fitted_model, user_ids, historical_item_list, k, allowed_product_ids)

    df_subm = pd.DataFrame({"visit_id": visit_ids, "product_ids": recoms, "scores": scores})
    df_subm["product_ids"] = df_subm["product_ids"].apply(lambda x: " ".join(x))
    return df_subm


def create_gt_submission(df_test):
    return (
        df_test
        .groupby("visit_id", sort=False)["product_id"]
        .agg(lambda x: " ".join(x.astype(str)))
        .reset_index()
    )


def postprocess_submission(df_subm, df_test):
    df_subm["product_ids"] = df_subm["product_ids"].str.split(" ")
    df_subm = df_subm.explode(["product_ids", "scores"])
    df_subm = df_subm.rename(columns={"product_ids": "product_id"})

    df_test = df_test[["visit_id", "product_id"]].drop_duplicates()
    df_test["label"] = 1

    df_subm = pd.merge(df_subm, df_test, how="left", on=["visit_id", "product_id"])
    df_subm["label"] = df_subm["label"].fillna(0)

    return df_subm


def evaluate_submission(df_subm, df_gt, n_list):
    def recall_at_k(rec, gt, k):
        if len(gt) == 0:
            return 0.0
        return len(set(rec[:k]) & set(gt)) / len(set(gt))

    df_gt = df_gt.copy()
    df_subm = df_subm.copy()
    df_subm["rec_list"] = df_subm["product_ids"].str.split()
    df_gt["gt_list"] = df_gt["product_id"].str.split()

    df = df_subm.merge(df_gt, on="visit_id", how="inner")

    for n in n_list:
        df["recall"] = df.apply(
            lambda x: recall_at_k(x["rec_list"], x["gt_list"], k=n),
            axis=1
        )
        print(f"recall@{n}: {df['recall'].mean():.4f}")


max_row_per_session = 20
n_candidates = 50
split_ratio = 0.50
visit_ids = get_test_visit_ids()
allowed_product_ids = get_allowed_product_ids()

df_hits = get_hits_data()
df_visits = get_visits_data()
df_data = create_recom_data(df_hits, df_visits)
df_data = add_start_token(df_data)
df_train, df_valid, df_test = get_data_splits(df_data, split_ratio)

df_train.to_csv("valid_historical.csv", index=False)

print(
    f"Train samples: {len(df_train)}\n"
    f"Train users: {df_train['client_id'].nunique()}\n"
    f"Train sessions: {df_train['visit_id'].nunique()}\n"
    f"Train items: {df_train['product_id'].nunique()}\n"
)

print(
    f"Valid samples: {len(df_valid)}\n"
    f"Valid users: {df_valid['client_id'].nunique()}\n"
    f"Valid sessions: {df_valid['visit_id'].nunique()}\n"
    f"Valid items: {df_valid['product_id'].nunique()}\n"
)

print(
    f"Test samples: {len(df_test)}\n"
    f"Test users: {df_test['client_id'].nunique()}\n"
    f"Test sessions: {df_test['visit_id'].nunique()}\n"
    f"Test items: {df_test['product_id'].nunique()}\n"
)

model = get_fitted_model(df_train, max_row_per_session)
allowed_product_ids = get_allowed_product_ids()

subm_valid = create_submission(
    fitted_model=model,
    hist_data=df_train,
    visit_ids=df_valid["visit_id"].unique(),
    k=n_candidates,
    allowed_product_ids=allowed_product_ids,
)
subm_gt_valid = create_gt_submission(df_valid)

evaluate_submission(subm_valid, subm_gt_valid, [6, n_candidates])
subm_valid = postprocess_submission(subm_valid, df_valid)

subm_valid.to_csv("valid_prediction.csv", index=False)
subm_gt_valid.to_csv("valid_gt.csv", index=False)
print(
    f"Valid data size: {len(subm_valid)}\n"
    f"Valid data pos ratio: {subm_valid['label'].mean():.4f}\n"
)

subm_test = create_submission(
    fitted_model=model,
    hist_data=df_train,
    visit_ids=df_test["visit_id"].unique(),
    k=n_candidates,
    allowed_product_ids=allowed_product_ids,
)
subm_gt_test = create_gt_submission(df_test)

evaluate_submission(subm_test, subm_gt_test, [6, n_candidates])
subm_test = postprocess_submission(subm_test, df_test)

subm_test.to_csv("test_prediction.csv", index=False)
subm_gt_test.to_csv("test_gt.csv", index=False)

print(
    f"Test data size: {len(subm_test)}\n"
    f"Test data pos ratio: {subm_test['label'].mean():.4f}\n"
)

df_data.to_csv("infer_historical.csv", index=False)

print(
    f"Infer Train samples: {len(df_data)}\n"
    f"Infer Train users: {df_data['client_id'].nunique()}\n"
    f"Infer Train sessions: {df_data['visit_id'].nunique()}\n"
    f"Infer Train items: {df_data['product_id'].nunique()}\n"
)

model = get_fitted_model(df_data, max_row_per_session)

subm_infer = create_submission(
    fitted_model=model,
    hist_data=df_data,
    visit_ids=visit_ids,
    k=n_candidates,
    allowed_product_ids=allowed_product_ids,
)

subm_infer = postprocess_submission(subm_infer, df_data)
subm_infer.to_csv("infer_prediction.csv", index=False)
print(
    f"Infer full output data size: {len(subm_infer)}\n"
    f"Infer full output data pos ratio: {subm_infer['label'].mean():.4f}\n"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 17.6 MB/s eta 0:00:00
Train samples: 1230371
Train users: 188131
Train sessions: 276162
Train items: 991

Valid samples: 17188
Valid users: 7259
Valid sessions: 13114
Valid items: 486

Test samples: 812
Test users: 566
Test sessions: 566
Test items: 216



  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 13114/13114 [00:11<00:00, 1102.38it/s]


recall@6: 0.3132
recall@50: 0.7088
Valid data size: 655700
Valid data pos ratio: 0.0185



100%|██████████| 566/566 [00:00<00:00, 1002.25it/s]


recall@6: 0.4111
recall@50: 0.7970
Test data size: 28300
Test data pos ratio: 0.0228

Infer Train samples: 1295242
Infer Train users: 188131
Infer Train sessions: 276162
Infer Train items: 1056



  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 1310/1310 [00:01<00:00, 765.28it/s]


Infer full output data size: 65500
Infer full output data pos ratio: 0.0000



### Ranking Stage

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import catboost as cb
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings("ignore")

root = "/kaggle/input/rental-product-recommendation-system"

latin2ru ={
    # Strollers
    "kolyaski": "Коляски",
    "progulochnye-kolyaski": "Прогулочные коляски",
    "kolyaski-dlya-puteshestviy": "Коляски YoYo",
    "kolyaski-yoyo": "Коляски YoYo",
    "kolyaski-dlya-novorozhdennyh-lyulki": "Коляски YoYo",

    # Car seats
    "avtokresla": "Автокресла",
    "detskie-avtokresla": "Автокресла, автолюльки",
    "avtokresla-avtolyulki": "Автокресла для новорождённых",
    "avtokresla-dlya-novorozhdyonnyh": "Автокресла для новорождённых",
    "avtokresla-9-36-kg": "Автокресла 9-36 кг",
    "velokresla": "Велокресла",

    # Furniture / sleep
    "krovatki-manezhi": "Кроватки, манежи",
    "manezhi-i-krovatki": "Кроватки, манежи",
    "bedroom": "Кроватки, манежи",
    "kokony-dlya-novorozhdennyh": "Коконы для новорожденных",
    "kokon-dlya-novorozhdennyh": "Коконы для новорожденных",

    # Feeding
    "stulchiki-dlya-kormleniya": "Стульчики для кормления",
    "stul-chiki-dlya-kormleniya": "Стульчики для кормления",
    "molokootsosy": "Молокоотсосы Medela",
    "molokootsosy-medela": "Молокоотсосы Medela",

    # Baby movement
    "hodunki": "Классические ходунки",
    "klassicheskie-hodunki": "Классические ходунки",
    "hodunki-katalki": "Ходунки-каталки",
    "hodunkikatalki": "Ходунки-каталки",
    "katalki": "Каталки",
    "begovely": "Беговелы",
    "samokaty": "Самокаты",
    "velosipedy": "Велосипеды",
    "kachalki": "Качалки",
    "kacheli-i-kachalki": "Качалки",
    "elektrokacheli": "Электрокачели",
    "elektro-kacheli": "Электрокачели",
    "shezlongi": "Шезлонги",
    "shezlongi-detskie-lyulki": "Шезлонги",

    # Toys
    "igrushki": "Игрушки",
    "konstruktory": "Конструкторы",
    "sortery": "Сортеры и пирамидки",
    "sortery-i-piramidki": "Сортеры и пирамидки",
    "razvivayuschie-kovriki": "Развивающие коврики",
    "muzykalnye-igrushki": "Музыкальные игрушки",
    "muzykal-nye-igrushki": "Музыкальные игрушки",
    "muzykalnye-instrumenty": "Музыкальные инструменты",
    "muzykalnye-stoliki": "Музыкальные столики",
    "razvivayuschie-stoliki": "Музыкальные столики",
    "mashinki-ruli-i-garazhi": "Машинки и гаражи",
    "parkovki-i-garazhi": "Машинки и гаражи",

    # Play & activity
    "igrovye-tsentry-i-kompleksy": "Игровые центры и комплексы",
    "multicentry": "Игровые центры и комплексы",
    "podvizhnye-igry": "Игровые центры и комплексы",
    "igrovye-paneli": "Игровые панели и бизиборды",
    "bizibordy": "Игровые панели и бизиборды",
    "busyboard": "Игровые панели и бизиборды",
    "prygunki": "Прыгунки",

    # Sports & outdoor
    "sportkompleksy": "Спортивные комплексы",
    "sportivnye-kompleksy": "Спортивные комплексы",
    "complex": "Спортивные комплексы",
    "aksessuary-k-sportkompleksam": "Аксессуары к спорткомплексам",
    "gorki": "Горки",
    "batuty": "Батуты",
    "suhie-basseyny": "Сухие бассейны",

    # Safety & monitoring
    "videonyani": "Видеоняни",
    "videonyani-prokat-radionyani": "Видеоняни",
    "radionyani": "Видеоняни",
    "ograzhdeniya": "Ограждения",
    "playpens": "Ограждения",

    # Bath & care
    "vannochki-dlya-kupaniya": "Ванночки для купания",
    "vsyo-dlya-kupaniya": "Ванночки для купания",
    "igrushki-dlya-kupaniya": "Игрушки для ванной",

    # Travel & storage
    "chemodany": "Чемоданы и рюкзаки",
    "chemodany-i-ryukzaki": "Чемоданы и рюкзаки",

    # Medical
    "detskie-vesy": "Весы",
    "vesy": "Весы",
    "vesy-sasha": "Весы Саша",
    "meditsinskie-tovary": "Медицинские товары"
}

ru2super = {
    # Strollers
    "Коляски": "strollers",
    "Прогулочные коляски": "strollers",
    "Коляски YoYo": "strollers",

    # Car seats
    "Автокресла": "car_seats",
    "Автокресла, автолюльки": "car_seats",
    "Автокресла для новорождённых": "car_seats",
    "Автокресла 9-36 кг": "car_seats",
    "Велокресла": "car_seats",

    # Furniture / sleep
    "Кроватки, манежи": "furniture_sleep",
    "Коконы для новорожденных": "furniture_sleep",

    # Feeding
    "Стульчики для кормления": "feeding",
    "Молокоотсосы Medela": "feeding",

    # Baby movement
    "Классические ходунки": "baby_movement",
    "Ходунки-каталки": "baby_movement",
    "Каталки": "baby_movement",
    "Беговелы": "baby_movement",
    "Самокаты": "baby_movement",
    "Велосипеды": "baby_movement",
    "Качалки": "baby_movement",
    "Электрокачели": "baby_movement",
    "Шезлонги": "baby_movement",

    # Toys
    "Игрушки": "toys",
    "Конструкторы": "toys",
    "Сортеры и пирамидки": "toys",
    "Развивающие коврики": "toys",
    "Музыкальные игрушки": "toys",
    "Музыкальные инструменты": "toys",
    "Музыкальные столики": "toys",
    "Машинки и гаражи": "toys",

    # Play & activity
    "Игровые центры и комплексы": "play_activity",
    "Игровые панели и бизиборды": "play_activity",
    "Прыгунки": "play_activity",

    # Sports & outdoor
    "Спортивные комплексы": "sports_outdoor",
    "Аксессуары к спорткомплексам": "sports_outdoor",
    "Горки": "sports_outdoor",
    "Батуты": "sports_outdoor",
    "Сухие бассейны": "sports_outdoor",

    # Safety & monitoring
    "Видеоняни": "safety_monitoring",
    "Ограждения": "safety_monitoring",

    # Bath & care
    "Ванночки для купания": "bath_care",
    "Игрушки для ванной": "bath_care",

    # Travel & storage
    "Чемоданы и рюкзаки": "travel_storage",

    # Medical
    "Весы": "medical",
    "Весы Саша": "medical",
    "Медицинские товары": "medical",
}


def get_id_to_slug():
    return(
        pd.read_csv(
            f"{root}/new_site_products.csv",
            usecols=["id", "slug"],
            dtype=str,
        )
        .drop_duplicates(subset=["id"])
        .set_index("id")["slug"]
        .to_dict()
    )


def evaluate_submission(df_subm, df_gt, n):
    def recall_at_k(rec, gt, k):
        if len(gt) == 0:
            return 0.0
        return len(set(rec[:k]) & set(gt)) / len(set(gt))

    df_gt = df_gt.copy()
    df_subm = df_subm.copy()
    df_subm["rec_list"] = df_subm["product_id"].str.split()
    df_gt["gt_list"] = df_gt["product_id"].str.split()

    return (
        df_subm
        .merge(df_gt, on="visit_id", how="inner")
        .apply(
            lambda x: recall_at_k(x["rec_list"], x["gt_list"], k=n),
            axis=1
        ).mean()
    )


def get_product_info():
    dtype_mapper = {
        "id": str,
        "main_category": str,
        'price_per_period3days': float,
        'price_per_period_week': float,
        'price_per_period2weeks': float,
        'price_per_period3weeks': float,
        'price_per_period4weeks': float,
        'weight': float,
    }

    df = pd.read_csv(
        f"{root}/new_site_products.csv",
        usecols=list(dtype_mapper.keys()),
        dtype=dtype_mapper,
    )
    df = df.rename(columns={"id": "product_id"})
    df = df.drop_duplicates("product_id")

    return df


def get_visit_info():
    dtype_mapper = {
        "visit_id": str,
        "region_country": str,
        "traffic_source": str,
        "is_new_user": str,
    }

    df = pd.concat([
        pd.read_csv(
            f"{root}/metrika_visits.csv",
            usecols=list(dtype_mapper.keys()),
            dtype=dtype_mapper,
        ),
        pd.read_csv(
            f"{root}/metrika_visits_test.csv",
            usecols=list(dtype_mapper.keys()),
            dtype=dtype_mapper,
        ),
    ], axis=0)
    df = df.drop_duplicates(subset=["visit_id"])

    return df


def get_historical_info(filepath):
    df = pd.read_csv(filepath, dtype=str)
    df = df.sort_values("date_time")

    df["product_id_if_product"] = df["product_id"].where(
        df["page_type"].eq("PRODUCT")
    )

    df["slug_if_category"] = df["slug"].where(
        df["page_type"].eq("CATEGORY")
    )

    return (
        df
        .groupby("visit_id", sort=False, as_index=False)
        .agg(
            event_list=("product_id", list),
            last_event=("product_id", "last"),
            last_event_type=("page_type", "last"),
            last_product_event=("product_id_if_product", "last"),
            last_category_event=("slug_if_category", "last"),
        )
    )


def get_test_visit_ids():
    return pd.read_csv(
        f"{root}/metrika_visits_test.csv",
        usecols=['visit_id'],
        dtype=str
    )["visit_id"].unique()


def feature_eng(df, df_prod, df_visit, df_hist):
    df = pd.merge(df, df_hist, on="visit_id", how="left")
    df = pd.merge(df, df_visit, on="visit_id", how="left")
    df = pd.merge(df, df_prod, on="product_id", how="left")

    prod2cat = df_prod.set_index("product_id")["main_category"]
    id2slug = get_id_to_slug()

    df["slug"] = df["product_id"].map(id2slug)
    df["slug_last_product_event"] = df["last_product_event"].map(id2slug)

    df["last_product_event_category"] = df["last_product_event"].map(prod2cat)
    df["last_category_event"] = df["last_category_event"].map(latin2ru)

    df["main_super_category"] = df["main_category"].map(ru2super)
    df["last_product_event_super_category"] = df["last_product_event_category"].map(ru2super)
    df["last_super_category_event"] = df["last_category_event"].map(ru2super)

    df["is_same_product_category"] = df["main_category"] == df["last_product_event_category"]
    df["is_same_category"] = df["main_category"] == df["last_category_event"]

    df["is_same_product_super_category"] = df["main_super_category"] == df["last_product_event_super_category"]
    df["is_same_super_category"] = df["main_super_category"] == df["last_super_category_event"]

    drop_columns = [
        "slug",
        "slug_last_product_event",
        "last_event",
        "last_product_event",
        "last_product_event_category",
        "last_product_event_super_category",
        "last_category_event",
        "last_super_category_event",
        "event_list",
    ]

    rank_columns = [
        'price_per_period3days',
        'price_per_period_week',
        'price_per_period2weeks',
        'price_per_period3weeks',
        'price_per_period4weeks',
        "scores",
    ]

    df = df.drop(columns=drop_columns)
    df[rank_columns] = df.groupby("visit_id")[rank_columns].rank()

    return df


def create_submission(g_test, y_pred):
    df_subm = g_test[["visit_id", "product_id"]].copy()
    df_subm["pred"] = y_pred

    df_subm = (
        df_subm
        .sort_values("pred", ascending=False)
        .groupby("visit_id", as_index=False)
        .head(6)
    )
    df_subm = df_subm.groupby("visit_id", as_index=False)["product_id"].apply(" ".join)
    return df_subm


def compare_with(df_subm, filepath):
    df_subm = df_subm.copy()
    df_subm = df_subm.set_index("visit_id")
    df_comp = pd.read_csv(
        filepath,
        dtype=str,
        index_col="visit_id",
    )

    df_comp = df_comp[df_comp.index.isin(df_subm.index)]
    df_subm = df_subm[df_subm.index.isin(df_comp.index)]

    matches = []
    for row1, row2 in zip(df_subm["product_id"], df_comp.loc[df_subm.index, "product_ids"]):
        row1 = row1.split(" ")
        row2 = row2.split(" ")
        match = len(set(row1) & set(row2))
        matches.append(match)

    print(f"*** {filepath} ***")
    print(f"Match ratio: {np.mean(matches):.4f}")
    print(f"Submission_new length: {len(df_subm)}")
    print(f"Submission_best length: {len(df_comp)}\n")


def get_id_to_slug():
    return(
        pd.read_csv(
            f"{root}/new_site_products.csv",
            usecols=["id", "slug"],
            dtype=str,
        )
        .drop_duplicates(subset=["id"])
        .set_index("id")["slug"]
        .to_dict()
    )


df_train = pd.read_csv(
    "valid_prediction.csv",
    dtype={"visit_id": str, "product_id":str, "scores": float, "label": int},
)
df_test = pd.read_csv(
    "test_prediction.csv",
    dtype={"visit_id": str, "product_id":str, "scores": float, "label": int},
)
df_infer = pd.read_csv(
    "infer_prediction.csv",
    dtype={"visit_id": str, "product_id":str, "scores": float, "label": int},
)
df_gt_train = pd.read_csv(
    "valid_gt.csv",
    dtype={"visit_id": str, "product_id":str},
)
df_gt_test = pd.read_csv(
    "test_gt.csv",
    dtype={"visit_id": str, "product_id":str},
)

df_train = df_train[df_train.groupby("visit_id")["label"].transform("max") > 0.5]

df_prod = get_product_info()
df_visit = get_visit_info()

df_hist_1 = get_historical_info("valid_historical.csv")
df_hist_2 = get_historical_info("infer_historical.csv")

df_train = feature_eng(df_train, df_prod, df_visit, df_hist_1)
df_test = feature_eng(df_test, df_prod, df_visit, df_hist_1)
df_infer = feature_eng(df_infer, df_prod, df_visit, df_hist_2)

X_data = df_train.drop(columns=["visit_id", "product_id", "label"])
y_data = df_train["label"]
i_data = df_train[["visit_id", "product_id"]]

X_test = df_test.drop(columns=["visit_id", "product_id", "label"])
y_test = df_test["label"]
i_test = df_test[["visit_id", "product_id"]]

X_infer = df_infer.drop(columns=["visit_id", "product_id", "label"])
i_infer = df_infer[["visit_id", "product_id"]]

encoder = OrdinalEncoder(
    unknown_value=-1,
    encoded_missing_value=-2,
    handle_unknown="use_encoded_value",
    dtype=np.int32,
    min_frequency=200,
)

cat_columns = X_data.select_dtypes(include=["object"]).columns.tolist()

X_data[cat_columns] = encoder.fit_transform(X_data[cat_columns])
X_data[cat_columns] = X_data[cat_columns].astype("category")

X_test[cat_columns] = encoder.transform(X_test[cat_columns])
X_test[cat_columns] = X_test[cat_columns].astype("category")

X_infer[cat_columns] = encoder.transform(X_infer[cat_columns])
X_infer[cat_columns] = X_infer[cat_columns].astype("category")

model = cb.CatBoostRanker(
    loss_function="YetiRankPairwise",
    eval_metric="NDCG:top=6",
    iterations=500,
    metric_period=100,
    learning_rate=0.05,
    depth=5,
    bootstrap_type="Bernoulli",
    subsample=0.5,
    sampling_unit="Group",
    task_type="GPU",
    random_state=42,
    verbose=0,
).fit(
    cb.Pool(
        data=X_data,
        label=y_data,
        group_id=i_data["visit_id"],
        cat_features=cat_columns,
    )
)

y_pred = X_test["scores"]
df_subm = create_submission(i_test, y_pred)
score = evaluate_submission(df_subm, df_gt_test, 6)
print(f"gru4rec recall@6: {score:.4f}")

y_pred = model.predict(X_test)
df_subm = create_submission(i_test, y_pred)
score = evaluate_submission(df_subm, df_gt_test, 6)
print(f"catboost recall@6: {score:.4f}")

cold_start_sessions = df_test.loc[
    df_test["last_event_type"].eq("START_NEW"), "visit_id"].unique()

cold_start_recom = df_subm.loc[
    df_subm["visit_id"].isin(cold_start_sessions), "product_id"]

cold_start_top6_recom = (
    cold_start_recom
    .str.split(" ")
    .explode()
    .value_counts()
    .nlargest(6)
    .index.tolist()
)
cold_start_top6_recom = " ".join(cold_start_top6_recom)
print("cold start recom: ", cold_start_top6_recom)

y_pred = model.predict(X_infer)

df_subm = create_submission(i_infer, y_pred)
df_subm = df_subm.rename(columns={"product_id": "product_ids"})
df_subm = df_subm.set_index("visit_id")

test_visit_ids = get_test_visit_ids()
missing_visit_ids = list(set(test_visit_ids) - set(df_subm.index))

df_subm = pd.concat([
    df_subm,
    pd.Series(index=missing_visit_ids, data=cold_start_top6_recom, name="product_ids")],
    axis=0
)

df_subm = df_subm.reset_index()
df_subm = df_subm.rename(columns={"index": "visit_id"})
df_subm.to_csv("submission.csv", index=False)
df_subm.head()

Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=6;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


gru4rec recall@6: 0.4111
catboost recall@6: 0.4319
cold start recom:  463480493 463480693 495400618 463480429 463480225 463480242


,visit_id,product_ids
0,0000913076882209575,463480355 495401665 463480484 463480359 463480...
1,0002268743573412674,463480322 495252594 495251312 463480553 495255...
2,0017209440033371282,463480225 495277614 463480429 463480224 463480...
3,0028175944123250496,1639075793 1640108065 1640113489 1640109097 46...
4,0031343418629767038,495252594 463480694 495251312 463480553 495251...
